## Evaluation with Pre-Built Foundry Evaluators

### Installing Utilities and Libraries

In [ ]:
%pip install azure-ai-projects==2.7.0 openai==1.109.1 python-dotenv azure-identity

### Setting up the Environment Variables

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("CHAT_MODEL_DEPLOYMENT_NAME")

### Creating the Foundry Project Client

In [ ]:
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential()
)

### Upload the Dataset to Foundry Project

In [ ]:
# Upload a local JSONL file tp your Foundry Project
data_id = project_client.datasets.upload_file(
    name="rag-eval-dataset",
    version="v1",
    file_path="./rag_eval_dataset.jsonl",
).id

### Define the Data Schema Mapping and Evaluators

In [ ]:
from openai.types.eval_create_params import DataSourceConfigCustom
from azure.ai.projects.models import (
    TestingCriterionAzureAIEvaluator,
)

data_source_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "query": {"type": "string"},
            "response": {"type": "string"},
            "ground_truth": {"type": "string"},
        },
        "required": ["query", "response", "ground_truth"],
    },
)

testing_criteria = [
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="coherence",
        evaluator_name="builtin.coherence",
        initialization_parameters={"model": model_deployment_name},
        data_mapping={
            "query": "{{item.query}}",
            "response": "{{item.response}}",
        },
    ),

    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="groundedness",
        evaluator_name="builtin.groundedness",
        initialization_parameters={
            "model": model_deployment_name
        },
        data_mapping={
            "query": "{{item.query}}",
            "response": "{{item.response}}",
            "context": "{{item.ground_truth}}",
        },
    ),

    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="violence",
        evaluator_name="builtin.violence",
        data_mapping={
            "query": "{{item.query}}",
            "response": "{{item.response}}",
        },
    )
]

### Create Evaluation and Run

In [ ]:
from openai.types.evals.create_eval_jsonl_run_data_source_param import (
    CreateEvalJSONLRunDataSourceParam,
    SourceFileID,
)

# create the openai client
openai_client = project_client.get_openai_client()

# Create the evaluation
eval_object = openai_client.evals.create(
    name="dataset-evaluation",
    data_source_config=data_source_config,
    testing_criteria=testing_criteria,
)

# Create a run using the uploaded dataset
eval_run = openai_client.evals.runs.create(
    eval_id=eval_object.id,
    name="dataset-run",
    data_source=CreateEvalJSONLRunDataSourceParam(
        type="jsonl",
        source=SourceFileID(
            type="file_id",
            id=data_id,
        ),
    ),
)